# Survived データセット探索

このノートブックでは、Survived（タイタニック号生存予測）データセットの探索的データ分析（EDA）を行います。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# スタイル設定（フォント設定の前に実行）
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 日本語フォント設定（自動検出）
japanese_fonts = [
    'MS Gothic',        # Windows
    'Yu Gothic',        # Windows
    'Meiryo',           # Windows
    'Hiragino Sans',    # macOS
    'IPAexGothic',      # Linux
    'Noto Sans CJK JP'  # Linux
]

# 利用可能なフォントを検索
available_fonts = [f.name for f in fm.fontManager.ttflist]
selected_font = None

for font in japanese_fonts:
    if font in available_fonts:
        selected_font = font
        break

if selected_font:
    plt.rcParams['font.family'] = selected_font
    print(f"日本語フォント: {selected_font} を使用します")
else:
    print("警告: 日本語フォントが見つかりませんでした。デフォルトフォントを使用します。")

# マイナス記号が文字化けしないように設定
plt.rcParams['axes.unicode_minus'] = False

print(f"現在のフォント設定: {plt.rcParams['font.family']}")

日本語フォント: MS Gothic を使用します
現在のフォント設定: ['MS Gothic']


## 1. データ読み込み

In [2]:
# Survived データセットを読み込む
df = pd.read_csv('../data/Survived.csv')

print(f"データ件数: {len(df)} 件")
print(f"特徴量数: {len(df.columns) - 1} 個")
print("\n最初の5行:")
df.head()

データ件数: 891 件
特徴量数: 10 個

最初の5行:


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,male,35.0,0,0,373450,8.0500,NaN,S


## 2. データの基本統計

In [3]:
# 基本統計量
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [4]:
# カテゴリカル変数の統計
print("性別の分布:")
print(df['Sex'].value_counts())
print("\n客室クラスの分布:")
print(df['Pclass'].value_counts())
print("\n生存状況:")
print(df['Survived'].value_counts())

性別の分布:
Sex
male      577
female    314
Name: count, dtype: int64

客室クラスの分布:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

生存状況:
Survived
0    549
1    342
Name: count, dtype: int64


## 3. 欠損値の確認

In [ ]:
# 欠損値の数を確認
missing_values = df.isnull().sum()
print("欠損値の数:")
print(missing_values)
print(f"\n欠損値の割合:")
print((missing_values / len(df) * 100).round(2))

In [ ]:
# 欠損値の可視化
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('欠損値の分布（黄色 = 欠損）')
plt.xlabel('列名')
plt.tight_layout()
plt.show()

## 4. 生存率の分析

In [ ]:
# 全体の生存率
survival_rate = df['Survived'].mean()
print(f"全体の生存率: {survival_rate * 100:.2f}%")

# 生存者数と死亡者数
survived_counts = df['Survived'].value_counts()
print(f"\n生存者数: {survived_counts[1]} 人")
print(f"死亡者数: {survived_counts[0]} 人")

# 可視化
plt.figure(figsize=(8, 6))
plt.bar(['死亡 (0)', '生存 (1)'], survived_counts.values, color=['#d62728', '#2ca02c'])
plt.xlabel('生存状況')
plt.ylabel('人数')
plt.title('生存者と死亡者の分布')
plt.grid(axis='y', alpha=0.3)

# パーセンテージを表示
for i, (label, value) in enumerate(zip(['死亡 (0)', '生存 (1)'], survived_counts.values)):
    plt.text(i, value + 10, f'{value} ({value/len(df)*100:.1f}%)', ha='center')

plt.tight_layout()
plt.show()

## 5. 性別による生存率

In [ ]:
# 性別ごとの生存率
sex_survival = df.groupby('Sex')['Survived'].mean()
print("性別ごとの生存率:")
print(sex_survival)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 生存率の棒グラフ
ax1 = axes[0]
sex_survival.plot(kind='bar', ax=ax1, color=['#ff9999', '#66b3ff'])
ax1.set_xlabel('性別')
ax1.set_ylabel('生存率')
ax1.set_title('性別ごとの生存率')
ax1.set_xticklabels(['女性', '男性'], rotation=0)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1)

# 生存者数の積み上げ棒グラフ
ax2 = axes[1]
sex_survived_crosstab = pd.crosstab(df['Sex'], df['Survived'])
sex_survived_crosstab.plot(kind='bar', stacked=True, ax=ax2, color=['#d62728', '#2ca02c'])
ax2.set_xlabel('性別')
ax2.set_ylabel('人数')
ax2.set_title('性別ごとの生存者数')
ax2.set_xticklabels(['女性', '男性'], rotation=0)
ax2.legend(['死亡', '生存'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 客室クラスによる生存率

In [ ]:
# 客室クラスごとの生存率
pclass_survival = df.groupby('Pclass')['Survived'].mean()
print("客室クラスごとの生存率:")
print(pclass_survival)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 生存率の棒グラフ
ax1 = axes[0]
pclass_survival.plot(kind='bar', ax=ax1, color=['#ff9999', '#ffcc99', '#99ccff'])
ax1.set_xlabel('客室クラス')
ax1.set_ylabel('生存率')
ax1.set_title('客室クラスごとの生存率')
ax1.set_xticklabels(['1等', '2等', '3等'], rotation=0)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1)

# 生存者数の積み上げ棒グラフ
ax2 = axes[1]
pclass_survived_crosstab = pd.crosstab(df['Pclass'], df['Survived'])
pclass_survived_crosstab.plot(kind='bar', stacked=True, ax=ax2, color=['#d62728', '#2ca02c'])
ax2.set_xlabel('客室クラス')
ax2.set_ylabel('人数')
ax2.set_title('客室クラスごとの生存者数')
ax2.set_xticklabels(['1等', '2等', '3等'], rotation=0)
ax2.legend(['死亡', '生存'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 年齢分布と生存

In [ ]:
# 年齢の分布
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 年齢のヒストグラム
ax1 = axes[0]
df['Age'].dropna().hist(bins=30, ax=ax1, edgecolor='black', alpha=0.7)
ax1.set_xlabel('年齢')
ax1.set_ylabel('人数')
ax1.set_title('年齢の分布')
ax1.axvline(df['Age'].mean(), color='red', linestyle='--', label=f'平均: {df["Age"].mean():.1f}歳')
ax1.axvline(df['Age'].median(), color='green', linestyle='--', label=f'中央値: {df["Age"].median():.1f}歳')
ax1.legend()

# 生存状況別の年齢分布
ax2 = axes[1]
df[df['Survived'] == 1]['Age'].dropna().hist(bins=30, ax=ax2, alpha=0.6, label='生存', color='green', edgecolor='black')
df[df['Survived'] == 0]['Age'].dropna().hist(bins=30, ax=ax2, alpha=0.6, label='死亡', color='red', edgecolor='black')
ax2.set_xlabel('年齢')
ax2.set_ylabel('人数')
ax2.set_title('生存状況別の年齢分布')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. 運賃と生存

In [ ]:
# 運賃の分布
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 運賃のヒストグラム（対数スケール）
ax1 = axes[0]
df['Fare'].hist(bins=50, ax=ax1, edgecolor='black', alpha=0.7)
ax1.set_xlabel('運賃')
ax1.set_ylabel('人数')
ax1.set_title('運賃の分布')
ax1.set_yscale('log')

# 生存状況別の運賃の箱ひげ図
ax2 = axes[1]
df.boxplot(column='Fare', by='Survived', ax=ax2)
ax2.set_xlabel('生存状況 (0: 死亡, 1: 生存)')
ax2.set_ylabel('運賃')
ax2.set_title('生存状況別の運賃')
plt.suptitle('')  # デフォルトのタイトルを削除

plt.tight_layout()
plt.show()

## 9. 家族サイズと生存

In [ ]:
# 家族サイズを計算（自分 + SibSp + Parch）
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 家族サイズごとの生存率
family_survival = df.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
print("家族サイズごとの生存率:")
print(family_survival)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 家族サイズの分布
ax1 = axes[0]
df['FamilySize'].value_counts().sort_index().plot(kind='bar', ax=ax1, color='skyblue', edgecolor='black')
ax1.set_xlabel('家族サイズ')
ax1.set_ylabel('人数')
ax1.set_title('家族サイズの分布')
ax1.grid(axis='y', alpha=0.3)

# 家族サイズごとの生存率
ax2 = axes[1]
family_survival['mean'].plot(kind='bar', ax=ax2, color='green', edgecolor='black')
ax2.set_xlabel('家族サイズ')
ax2.set_ylabel('生存率')
ax2.set_title('家族サイズごとの生存率')
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. 相関分析

In [ ]:
# 数値特徴量のみを選択して相関を計算
# Sex を数値化（male=1, female=0）
df_numeric = df.copy()
df_numeric['Sex_encoded'] = (df_numeric['Sex'] == 'male').astype(int)

# 相関行列
correlation_matrix = df_numeric[['Survived', 'Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('特徴量の相関行列')
plt.tight_layout()
plt.show()

In [ ]:
# Survived との相関係数（絶対値の大きい順）
survived_correlation = correlation_matrix['Survived'].drop('Survived').abs().sort_values(ascending=False)
print("生存との相関係数（絶対値）:")
print(survived_correlation)

plt.figure(figsize=(10, 6))
survived_correlation.plot(kind='barh', color='steelblue')
plt.xlabel('相関係数の絶対値')
plt.title('各特徴量と生存の相関')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 11. まとめ

### 主な観察結果:

1. **全体の生存率**:
   - 約38%の乗客が生存
   - 死亡者の方が多い（クラス不均衡あり）

2. **性別の影響**:
   - 女性の生存率は約74%
   - 男性の生存率は約19%
   - **性別が最も強い予測因子**

3. **客室クラスの影響**:
   - 1等客室: 約63%生存
   - 2等客室: 約47%生存
   - 3等客室: 約24%生存
   - **社会階級が生存率に大きく影響**

4. **年齢の影響**:
   - 子供の生存率が比較的高い
   - 欠損値が約20%存在

5. **運賃の影響**:
   - 高い運賃を払った乗客の生存率が高い傾向
   - 客室クラスと相関が高い

6. **家族サイズの影響**:
   - 2-4人の家族サイズで生存率が高い
   - 単独または大家族は生存率が低い

7. **予測の見通し**:
   - Decision Tree で約80%の精度を達成
   - Sex と Pclass が主要な予測因子
   - クラス不均衡に注意が必要